## PART D: InfoNCE on BERT with Lambda sweep
### Premise:
1. Q1: Does InfoNCE improve results on BERT?
- InfoNCE is the missing model in the BERT+Roberta eval - Considering that InfoNCE was the best performer in the DistilBERT experiment, running the same on BERT will add comparable datapoints and the contrastive learning paradigm might improve BERT MI reg results.
2. Q2: Is there an optimal lambda for InfoNCE + BERT that improves reasoning and flips the previously contradictory results on CKA and Cosine?
- Varying lambda on the BERT experiment will help test the hypothesis that the different backbone strategy in BERT(NSP) might be clashing with MI and that lowering the MI weight on BERT might produce better results or that there is an optimal lambda at which each strategy performs best on each model. Testing the best MI strategy InfoNCE (as of DistilBERT) will help study improvements over original lambda for comparison and also the effect of varying lambda on BERT.
3. This code can be used separately for InfoNCE on Roberta for more result data points and also to run more lambda sweeps on the different MI strategies and models.

### Experimental Setup
- 1 strategy (infonce) x 5 lambdas (0, 0.001, 0.01, 0.05, 0.1) x 1 model (BERT)
- Scoped down to 3 epochs (instead of 5) and train:6000 instead of 8000 with 1500 val and 500 contra.
- Each run (3 epochs) takes approx 15-20mins on T4 -> 5 runs ~ 1.5-2 hrs

### Interpretation
1. Lambda == 0 is the baseline setup - with weight 0  negating the MI effect
2. Lambda == 0.01 will correspond to the InfoNCE implementation that would have been compared with the original CKA and Cosine (if not for the consideration that this experiment has been scoped down). Nevertheless, it is still comparable against the case 1 baseline result.
3. Lambda < 0.01 will correspond to the second hypothesis, wherein, lowering the MI regularisation weight is expected to reduce interference with the BERT pretraining architecture and improve performance. Therefore lambda == 0.001 ought to produce better results (compared to cases 1 and 2).
4. In general, if MI with InfoNCE were a viable paradigm on BERT, performance will peak at the optimal lambda (this could be 0.01 but is expected to be <) and degrade on either side.


In [ ]:
#starting from the mitr_bert_roberta_boolq.ipynb nb

In [1]:
!pip install -q --upgrade "transformers>=4.36" datasets accelerate matplotlib seaborn tqdm
print("Done.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 79.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 99.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 11.9 MB/s eta 0:00:00
Done.


In [2]:
import os, json, random, warnings
from dataclasses import dataclass
from typing import Dict, List, Optional
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoModel, AutoTokenizer,
    BertModel, BertTokenizerFast,
    get_cosine_schedule_with_warmup,
)
from datasets import load_dataset
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")

PyTorch : 2.10.0+cu128
CUDA    : True


In [21]:
@dataclass
class Config:
    # -- Model --
    model_name: str  = "bert-base-uncased"
    num_labels: int  = 2
    # -- Training --
    batch_size: int       = 16
    eval_batch_size: int  = 32
    learning_rate: float  = 2e-5
    weight_decay: float   = 0.01
    num_epochs: int       = 3
    warmup_ratio: float   = 0.06
    max_grad_norm: float  = 1.0
    grad_accum_steps: int = 4

    # -- MITR --
    # mi_lambda: float     = 0.01
    mi_warmup_steps: int = 200
    club_hidden: int     = 384
    mi_strategy: str     = "infonce"

    # -- Data --
    dataset: str         = "boolq"
    max_length: int      = 256
    num_train: int       = 6000
    num_eval: int        = 1500
    num_contra: int      = 500
    num_workers: int     = 2

    use_fp16: bool        = True

    # # -- A100 --
    use_bf16: bool       = False
    compile_model: bool  = False

    seed: int            = 42

    # -- Output --
    output_dir: str      = "experiment_results"

#just for bert - add roberta for a complete eval
BACKBONES = [
    "bert-base-uncased",
]

#add baseline for a full eval - will add 5 runs for the lambda sweep
MI_STRATEGIES = [
    # "baseline",
    "infonce"
    ]

LAMBDAS = [0, 0.001, 0.01, 0.05, 0.1]

print("Backbones:", BACKBONES)
print("Strategies:", MI_STRATEGIES)
print("Lambdas:", LAMBDAS)

Backbones: ['bert-base-uncased']
Strategies: ['infonce']
Lambdas: [0, 0.001, 0.01, 0.05, 0.1]


In [22]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32       = True
torch.backends.cudnn.benchmark        = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if torch.cuda.is_available():
    if torch.cuda.is_bf16_supported():
        DTYPE = torch.bfloat16
        print(" bf16 (A100)")
    else:
        DTYPE = torch.float16
        print(" fp16 (T4)")
else:
    DTYPE = torch.float32


print(f"Device : {DEVICE}")
print(f"Dtype  : {DTYPE}")
if DEVICE.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"GPU    : {props.name}")
    print(f"VRAM   : {props.total_memory / 1e9:.1f} GB")


 bf16 (A100)
Device : cuda
Dtype  : torch.bfloat16
GPU    : Tesla T4
VRAM   : 15.6 GB


In [23]:
# -- Load BoolQ --

def load_boolq():
    ds = load_dataset("google/boolq")
    def fmt(ex):
        return {
            "text":     ex["question"] + " [SEP] " + ex["passage"][:400],
            "label":    int(ex["answer"]),
            "question": ex["question"],
        }
    train = ds["train"].map(fmt, remove_columns=ds["train"].column_names)
    val   = ds["validation"].map(fmt, remove_columns=ds["validation"].column_names)
    return train, val


rng = random.Random(42)

def subsample(dataset, n):
    if n < 0 or n >= len(dataset):
        return dataset
    return dataset.select(rng.sample(range(len(dataset)), n))


print("Loading BoolQ ...")
train_raw, val_raw = load_boolq()
train_raw = subsample(train_raw, 6000)
val_raw   = subsample(val_raw,   1500)

tl = [x["label"] for x in train_raw]
vl = [x["label"] for x in val_raw]
print(f"Train : {len(train_raw):,}  (True={sum(tl)}, False={len(tl)-sum(tl)})")
print(f"Val   : {len(val_raw):,}  (True={sum(vl)}, False={len(vl)-sum(vl)})")

Loading BoolQ ...
Train : 6,000  (True=3728, False=2272)
Val   : 1,500  (True=931, False=569)


In [24]:
# -- Contradiction pairs (same as DistilBERT experiments) --

_AUX = [
    ("is ",     "is not "),
    ("are ",    "are not "),
    ("was ",    "was not "),
    ("were ",   "were not "),
    ("does ",   "does not "),
    ("do ",     "do not "),
    ("did ",    "did not "),
    ("has ",    "has not "),
    ("have ",   "have not "),
    ("had ",    "had not "),
    ("can ",    "cannot "),
    ("could ",  "could not "),
    ("will ",   "will not "),
    ("would ",  "would not "),
    ("should ", "should not "),
]

def negate_question(q: str) -> Optional[str]:
    q = q.strip().rstrip("?").lower()
    for pos, neg in _AUX:
        if q.startswith(neg):
            return pos + q[len(neg):]
        if q.startswith(pos):
            return neg + q[len(pos):]
    return None


def create_contradiction_pairs(dataset, n_pairs: int) -> List[Dict]:
    pairs: List[Dict] = []
    for ex in dataset:
        q = ex.get("question", "").strip()
        if not q:
            continue
        q_neg = negate_question(q)
        if q_neg is None or q_neg.strip() == q.strip():
            continue
        text_fwd = ex["text"]
        if "[SEP]" in text_fwd:
            prefix   = text_fwd.split("[SEP]")[0]
            text_neg = prefix + "[SEP] " + q_neg
        else:
            text_neg = q_neg
        pairs.append({
            "text_forward":    text_fwd,
            "text_negated":    text_neg,
            "label_forward":   ex["label"],
            "label_negated":   1 - ex["label"],
            "question":        q,
            "negated_question": q_neg,
        })
        if len(pairs) >= n_pairs:
            break
    return pairs


contradiction_pairs = create_contradiction_pairs(val_raw, n_pairs=500)
print(f"Contradiction pairs: {len(contradiction_pairs):,}")


Contradiction pairs: 500


In [25]:
# -- Dataset classes --
# These accept an arbitrary tokenizer so they work with BERT, RoBERTa, etc.

class LogicDataset(Dataset):
    def __init__(self, data, tokenizer, max_length: int):
        enc = tokenizer(
            [ex["text"] for ex in data],
            max_length=max_length, padding="max_length",
            truncation=True, return_tensors="pt",
        )
        self.input_ids      = enc["input_ids"]
        self.attention_mask = enc["attention_mask"]
        self.labels         = torch.tensor([ex["label"] for ex in data], dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids":      self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "labels":         self.labels[idx],
        }


class ContradictionPairDataset(Dataset):
    def __init__(self, pairs: List[Dict], tokenizer, max_length: int):
        fwd = tokenizer([p["text_forward"] for p in pairs],
                        max_length=max_length, padding="max_length",
                        truncation=True, return_tensors="pt")
        neg = tokenizer([p["text_negated"]  for p in pairs],
                        max_length=max_length, padding="max_length",
                        truncation=True, return_tensors="pt")
        self.fwd_ids  = fwd["input_ids"];     self.fwd_mask = fwd["attention_mask"]
        self.neg_ids  = neg["input_ids"];     self.neg_mask = neg["attention_mask"]
        self.fwd_lbl  = torch.tensor([p["label_forward"] for p in pairs], dtype=torch.long)
        self.neg_lbl  = torch.tensor([p["label_negated"] for p in pairs], dtype=torch.long)

    def __len__(self):
        return len(self.fwd_lbl)

    def __getitem__(self, idx):
        return {
            "fwd_input_ids":      self.fwd_ids[idx],
            "fwd_attention_mask": self.fwd_mask[idx],
            "neg_input_ids":      self.neg_ids[idx],
            "neg_attention_mask": self.neg_mask[idx],
            "fwd_label":          self.fwd_lbl[idx],
            "neg_label":          self.neg_lbl[idx],
        }


print("Dataset classes defined.")


Dataset classes defined.


In [26]:


class InfoNCEMI(nn.Module):
    def __init__(self, x_dim, y_dim, hidden, temperature=0.07):
        super().__init__()
        self.proj_x = nn.Sequential(
            nn.Linear(x_dim, hidden),
            nn.LayerNorm(hidden),
            nn.GELU(),
            nn.Linear(hidden, hidden),
        )
        self.proj_y = nn.Sequential(
            nn.Linear(y_dim, hidden),
            nn.LayerNorm(hidden),
            nn.GELU(),
            nn.Linear(hidden, hidden),
        )
        self.temperature = temperature

    def forward(self, x, y):
        x_proj = F.normalize(self.proj_x(x.float()), dim=-1)
        y_proj = F.normalize(self.proj_y(y.float()), dim=-1)
        sim    = torch.matmul(x_proj, y_proj.T) / self.temperature
        labels = torch.arange(x.size(0), device=x.device)
        loss   = F.cross_entropy(sim, labels)
        mi     = torch.log(torch.tensor(float(x.size(0)), device=x.device)) - loss
        return mi.clamp(-20.0, 20.0)

print("InfoNCE defined.")

InfoNCE defined.


In [27]:
class BaselineClassifier(nn.Module):
    """Backbone-agnostic baseline. Uses AutoModel to support any encoder."""
    def __init__(self, model_name: str, num_labels: int = 2):
        super().__init__()
        self.encoder        = AutoModel.from_pretrained(model_name)
        hidden_size         = self.encoder.config.hidden_size
        self.pre_classifier = nn.Linear(hidden_size, hidden_size)
        self.classifier     = nn.Linear(hidden_size, num_labels)
        self.dropout        = nn.Dropout(0.1)

    def forward(self, input_ids, attention_mask, labels=None, is_training=False):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0]  # CLS token
        cls = self.dropout(F.relu(self.pre_classifier(cls)))
        logits = self.classifier(cls)

        result = {"logits": logits, "mi_loss": 0.0}
        if labels is not None:
            result["loss"] = F.cross_entropy(logits, labels)
        return result


class MITRInfoNCEClassifier(nn.Module):
    def __init__(self, cfg: Config, mi_lambda: float):
        super().__init__()
        self.cfg = cfg
        self.mi_lambda       = mi_lambda
        self.encoder        = AutoModel.from_pretrained(cfg.model_name)
        hidden_size         = self.encoder.config.hidden_size
        self.pre_classifier = nn.Linear(hidden_size, hidden_size)
        self.classifier     = nn.Linear(hidden_size, cfg.num_labels)
        self.dropout        = nn.Dropout(0.1)

        estimator_cls       = InfoNCEMI
        self.mi_estimator   = estimator_cls(hidden_size, hidden_size, cfg.club_hidden)
        self._step          = 0

    def _effective_lambda(self) -> float:
        if self._step >= self.cfg.mi_warmup_steps:
            return self.mi_lambda
        return self.mi_lambda * (self._step / max(1, self.cfg.mi_warmup_steps))

    def forward(self, input_ids, attention_mask, labels=None, is_training=False):
        out = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,
        )

        cls    = out.last_hidden_state[:, 0]
        cls    = self.dropout(F.relu(self.pre_classifier(cls)))
        logits = self.classifier(cls)

        result = {"logits": logits, "mi_loss": 0.0}

        if labels is not None:
            task_loss = F.cross_entropy(logits, labels)
            lam       = self._effective_lambda()

            if is_training and lam > 0.0:
                hs = out.hidden_states  # (embedding + N layers)

                # Residual diffs between consecutive layers
                diffs = []
                for i in range(len(hs) - 1):
                    d = (hs[i + 1] - hs[i]).mean(dim=1)  # mean-pool tokens
                    d = F.layer_norm(d, (d.size(-1),))
                    diffs.append(d)

                # MI between consecutive diffs
                mi_list = [
                    self.mi_estimator(diffs[i], diffs[i + 1])
                    for i in range(len(diffs) - 1)
                ]

                if mi_list:
                    mi_mean          = torch.stack(mi_list).mean()
                    result["mi_loss"] = mi_mean.item()
                    result["loss"]    = (1.0 - lam) * task_loss + lam * mi_mean
                else:
                    result["loss"] = task_loss
            else:
                result["loss"] = task_loss

            if is_training:
                self._step += 1

        return result


print("InfoNCE classifier defined.")
print("  BERT hidden states:    13 (emb + 12 layers) -> 11 diff pairs -> 10 MI terms")


InfoNCE classifier defined.
  BERT hidden states:    13 (emb + 12 layers) -> 11 diff pairs -> 10 MI terms


In [28]:

def build_optimizer_scheduler(model, train_loader, cfg):
    no_decay = {"bias", "LayerNorm.weight"}
    grouped  = [
        {"params": [p for n, p in model.named_parameters()
                    if p.requires_grad and not any(nd in n for nd in no_decay)],
         "weight_decay": cfg.weight_decay},
        {"params": [p for n, p in model.named_parameters()
                    if p.requires_grad and any(nd in n for nd in no_decay)],
         "weight_decay": 0.0},
    ]
    try:
        opt = torch.optim.AdamW(grouped, lr=cfg.learning_rate, fused=True)
    except TypeError:
        opt = torch.optim.AdamW(grouped, lr=cfg.learning_rate)

    total_steps  = len(train_loader) * cfg.num_epochs // cfg.grad_accum_steps
    warmup_steps = int(total_steps * cfg.warmup_ratio)
    sched        = get_cosine_schedule_with_warmup(opt, warmup_steps, total_steps)
    return opt, sched


def train_one_epoch(model, loader, optimizer, scheduler, device, dtype,
                    grad_accum=1, max_grad_norm=1.0, is_mitr=False):
    model.train()
    total_loss = total_mi = n = 0
    use_amp = (dtype != torch.float32)

    optimizer.zero_grad()
    for step, batch in enumerate(tqdm(loader, desc="  train", leave=False)):
        ids  = batch["input_ids"].to(device, non_blocking=True)
        mask = batch["attention_mask"].to(device, non_blocking=True)
        lbls = batch["labels"].to(device, non_blocking=True)

        with torch.autocast(device_type=device.type, dtype=dtype, enabled=use_amp):
            out = model(ids, mask, labels=lbls, is_training=is_mitr)

        (out["loss"] / grad_accum).backward()

        if (step + 1) % grad_accum == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            optimizer.step(); scheduler.step(); optimizer.zero_grad()

        total_loss += out["loss"].item()
        mi_val      = out.get("mi_loss", 0.0)
        total_mi   += mi_val if isinstance(mi_val, float) else float(mi_val)
        n          += 1

    return {"train_loss": total_loss / n, "train_mi_loss": total_mi / n}


@torch.no_grad()
def eval_accuracy(model, loader, device, dtype):
    model.eval()
    correct = total = 0; val_loss = 0.0
    use_amp = (dtype != torch.float32)

    for batch in loader:
        ids  = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        lbls = batch["labels"].to(device)
        with torch.autocast(device_type=device.type, dtype=dtype, enabled=use_amp):
            out = model(ids, mask, labels=lbls)
        preds    = out["logits"].argmax(-1)
        correct += (preds == lbls).sum().item()
        total   += lbls.size(0)
        if "loss" in out:
            val_loss += out["loss"].item()

    return {"accuracy": correct / total, "val_loss": val_loss / len(loader)}


@torch.no_grad()
def eval_contradiction_rate(model, pair_loader, device, dtype):
    model.eval()
    contradictions = fwd_correct = neg_correct = total = 0
    use_amp = (dtype != torch.float32)

    for batch in pair_loader:
        fwd_ids  = batch["fwd_input_ids"].to(device)
        fwd_mask = batch["fwd_attention_mask"].to(device)
        neg_ids  = batch["neg_input_ids"].to(device)
        neg_mask = batch["neg_attention_mask"].to(device)
        fwd_lbl  = batch["fwd_label"].to(device)
        neg_lbl  = batch["neg_label"].to(device)

        with torch.autocast(device_type=device.type, dtype=dtype, enabled=use_amp):
            fwd_out = model(fwd_ids, fwd_mask)
            neg_out = model(neg_ids, neg_mask)

        fwd_pred = fwd_out["logits"].argmax(-1)
        neg_pred = neg_out["logits"].argmax(-1)

        contradictions += (fwd_pred == neg_pred).sum().item()
        fwd_correct    += (fwd_pred == fwd_lbl).sum().item()
        neg_correct    += (neg_pred == neg_lbl).sum().item()
        total          += fwd_lbl.size(0)

    return {
        "contradiction_rate": contradictions / total,
        "consistency_rate":   1.0 - contradictions / total,
        "fwd_accuracy":       fwd_correct / total,
        "neg_accuracy":       neg_correct / total,
    }


print("Training utilities defined.")



Training utilities defined.


In [29]:
def run_experiment(name, model, cfg, is_mitr, train_loader, val_loader, pair_loader):
    print(f"\n{'='*60}")
    print(f"  {name}")
    print(f"{'='*60}")

    model = model.to(DEVICE)

    if cfg.compile_model and hasattr(torch, "compile") and not is_mitr:
        try:
            model = torch.compile(model, mode="default")
            print("  torch.compile() enabled")
        except Exception as e:
            print(f"  torch.compile() skipped: {e}")
    elif is_mitr:
        print("  torch.compile() skipped for MITR")

    opt, sched = build_optimizer_scheduler(model, train_loader, cfg)
    history    = defaultdict(list)

    for epoch in range(1, cfg.num_epochs + 1):
        tr  = train_one_epoch(model, train_loader, opt, sched,
                               DEVICE, DTYPE,
                               grad_accum=cfg.grad_accum_steps,
                               max_grad_norm=cfg.max_grad_norm,
                               is_mitr=is_mitr)
        val = eval_accuracy(model, val_loader, DEVICE, DTYPE)

        history["epoch"].append(epoch)
        history["train_loss"].append(tr["train_loss"])
        history["train_mi_loss"].append(tr["train_mi_loss"])
        history["val_loss"].append(val["val_loss"])
        history["val_accuracy"].append(val["accuracy"])

        print(f"  Ep {epoch}/{cfg.num_epochs}  "
              f"train={tr['train_loss']:.4f}  mi={tr['train_mi_loss']:.4f}  "
              f"val={val['val_loss']:.4f}  acc={val['accuracy']:.4f}")

    print("\n  Evaluating contradiction rate ...")
    contra = eval_contradiction_rate(model, pair_loader, DEVICE, DTYPE)
    print(f"  Contradiction rate : {contra['contradiction_rate']:.4f}")
    print(f"  Consistency  rate  : {contra['consistency_rate']:.4f}")

    # Free GPU memory before next experiment
    del model, opt, sched
    torch.cuda.empty_cache()

    return {
        "name":               name,
        "history":            dict(history),
        "final_accuracy":     history["val_accuracy"][-1],
        "final_val_loss":     history["val_loss"][-1],
        "contradiction_rate": contra["contradiction_rate"],
        "consistency_rate":   contra["consistency_rate"],
        "fwd_accuracy":       contra["fwd_accuracy"],
        "neg_accuracy":       contra["neg_accuracy"],
    }


print("Experiment runner defined.")


Experiment runner defined.


In [30]:
all_results = {}
os.makedirs("experiment_results", exist_ok=True)

_dl_kw = dict(
    pin_memory=(DEVICE.type == "cuda"),
    num_workers=2,
    persistent_workers=True,
)

for backbone in BACKBONES:
    print(f"\n\n{'#'*70}")
    print(f"#  BACKBONE: {backbone}")
    print(f"{'#'*70}")

    tok = AutoTokenizer.from_pretrained(backbone)

    train_dataset = LogicDataset(train_raw, tok, 256)
    val_dataset   = LogicDataset(val_raw,   tok, 256)
    pair_dataset  = ContradictionPairDataset(contradiction_pairs, tok, 256)

    train_loader = DataLoader(train_dataset, batch_size=32,  shuffle=True,  **_dl_kw)
    val_loader   = DataLoader(val_dataset,   batch_size=64,  shuffle=False, **_dl_kw)
    pair_loader  = DataLoader(pair_dataset,  batch_size=64,  shuffle=False, **_dl_kw)

    short_name = backbone.split("/")[-1]  # e.g. "bert-base-uncased"

    # # 1. Baseline
    # set_seed(42)
    # model = BaselineClassifier(backbone, num_labels=2)
    # cfg   = Config(model_name=backbone)
    # key   = f"{short_name}_baseline"
    # all_results[key] = run_experiment(
    #     f"Baseline {short_name}", model, cfg, is_mitr=False,
    #     train_loader=train_loader, val_loader=val_loader, pair_loader=pair_loader,
    # )

    # 2. MITR with InfoNCE /++
    for lmbda in LAMBDAS:
      for strategy in MI_STRATEGIES:
          key   = f"{short_name}_{strategy}_{lmbda}"
          set_seed(42)
          cfg = Config(model_name=backbone, mi_strategy=strategy)
          if lmbda == 0.0:
            # Baseline: no MI regularization
            model    = BaselineClassifier(backbone, num_labels=2)
            cfg   = Config(model_name=backbone)
            all_results[key] = run_experiment(
                f"MITR Baseline-{strategy.upper()} {short_name} {lmbda}", model, cfg, is_mitr=False,
                train_loader=train_loader, val_loader=val_loader, pair_loader=pair_loader,
            )
          else:
            model = MITRInfoNCEClassifier(cfg, lmbda)

            all_results[key] = run_experiment(
                f"MITR InfoNCE-{strategy.upper()} {short_name} {lmbda}", model, cfg, is_mitr=True,
                train_loader=train_loader, val_loader=val_loader, pair_loader=pair_loader,
            )

# Save all results
out_json = "experiment_results/results_bert_lambdas.json"
with open(out_json, "w") as fh:
    json.dump(all_results, fh, indent=2, default=float)
print(f"\nAll results saved -> {out_json}")



######################################################################
#  BACKBONE: bert-base-uncased
######################################################################


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



  MITR Baseline-INFONCE bert-base-uncased 0


  train:   0%|          | 0/188 [00:00<?, ?it/s]

...

  Ep 1/3  train=0.5046  mi=-1.7926  val=0.6592  acc=0.6207


  train:   0%|          | 0/188 [00:00<?, ?it/s]

  Ep 2/3  train=-0.7163  mi=-13.0864  val=0.6568  acc=0.6207


  train:   0%|          | 0/188 [00:00<?, ?it/s]

  Ep 3/3  train=-1.0451  mi=-16.3445  val=0.6550  acc=0.6207

  Evaluating contradiction rate ...
  Contradiction rate : 1.0000
  Consistency  rate  : 0.0000

All results saved -> experiment_results/results_bert_lambdas.json


In [53]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def plot_lambda_sweep_plotly(results):
    # ---- Parse + sort ----
    parsed = []
    for k, v in results.items():
        lam = float(k.split("_")[-1])
        parsed.append((lam, v))
    parsed = sorted(parsed, key=lambda x: x[0])

    lambdas = [p[0] for p in parsed]
    lambda_labels = [str(l) for l in lambdas]

    final_acc = [res["final_accuracy"] for _, res in parsed]
    contra = [res["contradiction_rate"] for _, res in parsed]

    # ---- Create 2x3 subplot ----
    fig = make_subplots(
        rows=2, cols=3,
        subplot_titles=[
            "Validation Accuracy vs Epoch",
            "Training Loss vs Epoch",
            "MI Loss vs Epoch",
            "Accuracy vs Contradiction",
            "Final Accuracy vs Lambda",
            "Contradiction Rate vs Lambda"
        ]
    )

    # =========================================================
    # 1. Validation Accuracy
    # =========================================================
    for lam, res in parsed:
        fig.add_trace(go.Scatter(
            x=res["history"]["epoch"],
            y=res["history"]["val_accuracy"],
            mode="lines+markers",
            name=f"λ={lam}"
        ), row=1, col=1)

    # =========================================================
    # 2. Training Loss
    # =========================================================
    for lam, res in parsed:
        fig.add_trace(go.Scatter(
            x=res["history"]["epoch"],
            y=res["history"]["train_loss"],
            mode="lines+markers",
            showlegend=False
        ), row=1, col=2)

    # =========================================================
    # 3. MI Loss
    # =========================================================
    for lam, res in parsed:
        fig.add_trace(go.Scatter(
            x=res["history"]["epoch"],
            y=res["history"]["train_mi_loss"],
            mode="lines+markers",
            showlegend=False
        ), row=1, col=3)

    # =========================================================
    # 4. Double Bar Plot (GREEN + MAROON)
    # =========================================================
    fig.add_trace(go.Bar(
        x=lambda_labels,
        y=final_acc,
        name="Accuracy",
        marker_color="green",
        text=[f"{a:.3f}" for a in final_acc],
        textposition="outside"
    ), row=2, col=1)

    fig.add_trace(go.Bar(
        x=lambda_labels,
        y=contra,
        name="Contradiction",
        marker_color="maroon",
        text=[f"{c:.3f}" for c in contra],
        textposition="outside"
    ), row=2, col=1)

    # =========================================================
    # 5. Accuracy-only Bar Plot
    # =========================================================
    fig.add_trace(go.Bar(
        x=lambda_labels,
        y=final_acc,
        width=0.4,
        marker_color="green",
        text=[f"{a:.3f}" for a in final_acc],
        textposition="outside",
        showlegend=False
    ), row=2, col=2)

    # =========================================================
    # 6. Contradiction-only Bar Plot
    # =========================================================
    fig.add_trace(go.Bar(
        x=lambda_labels,
        y=contra,
        width=0.4,
        marker_color="maroon",
        text=[f"{c:.3f}" for c in contra],
        textposition="outside",
        showlegend=False
    ), row=2, col=3)

    # =========================================================
    # Layout (NO GRIDLINES)
    # =========================================================
    fig.update_layout(
        height=800,
        width=1200,
        title="MITR InfoNCE Lambda Sweep (BERT)",
        template="simple_white",
        barmode="group"
    )

    # Remove ALL gridlines explicitly
    fig.update_xaxes(showgrid=False)
    fig.update_yaxes(showgrid=False)

    fig.write_html("infonce_lambda_plots.html")
    fig.show()


# Run
plot_lambda_sweep_plotly(all_results)